In [10]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
#import torch
#import torch.nn as nn
#from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity
from dynamic_similarities import build_statistical_similarity_graph, build_dynamic_similarity_graphs
import networkx as nx
import matplotlib.pyplot as plt
from plot import save_graph_plot, save_dynamic_graph_plots


In [11]:
DATA_PATH = '../../dataset/representative_items.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

NUM_ITEMS = 100
df = df[df['item_id'].isin(df['item_id'].unique()[:NUM_ITEMS])]

Loading data from ../../dataset/representative_items.feather...


In [12]:
# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)
ITEM_COL = 'item_id'

# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df['item_id'] = df['item_id'].astype(int) # or .astype(str)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]
print(len(y))

# Calandar-based features
# Ensure datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Base calendar parts
df["day_of_week"]  = df[DATE_COL].dt.dayofweek.astype(int)
df["day_of_month"] = df[DATE_COL].dt.day.astype(int)         
df["moy"]          = df[DATE_COL].dt.month.astype(int)-1      
df["doy"]          = df[DATE_COL].dt.dayofyear.astype(int)-1   
df["is_weekend"] = (
    (df[DATE_COL].dt.dayofweek == 5) |
    (df[DATE_COL].dt.dayofweek == 6)
).astype(int)

m0 = df[DATE_COL].dt.month - 1
df["month_sin"] = np.sin(2*np.pi*m0 / 12)
df["month_cos"] = np.cos(2*np.pi*m0 / 12)

df["doy"] = df[DATE_COL].dt.dayofyear # 1..365/366
doy0 = df["doy"] - 1
P = 366 # safe; or use 365 if you drop leap years
df["doy_sin"] = np.sin(2*np.pi*doy0 / P)
df["doy_cos"] = np.cos(2*np.pi*doy0 / P)

promo_types= [col for col in df.columns if col.startswith("promo_type_")] # Binary columns indicating presence of specific promotion types
promo_values= [col for col in df.columns if col.startswith("promo_value_")] # Numerical columns indicating the value of specific promotion types

calendar_cols = ["day_of_week", "day_of_month", "moy", "doy", "is_weekend"]
calendar_trigonometric_cols = ["month_sin", "month_cos", "doy_sin", "doy_cos"]
categorical_cols = ["cat_label", "sdep_label", "dept_label"]

df = df.sort_values(["item_id", DATE_COL]).reset_index(drop=True)

df

76049


,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,item_label,day_of_week,day_of_month,moy,doy,is_weekend,month_sin,month_cos,doy_sin,doy_cos
0,2021-01-23,676,4,refrig dsrts,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,representative,5,23,0,23,1,0.0,1.000000,0.368763,0.929523
1,2021-01-24,676,4,refrig dsrts,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,representative,6,24,0,24,1,0.0,1.000000,0.384665,0.923056
2,2021-01-25,676,4,refrig dsrts,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,representative,0,25,0,25,0,0.0,1.000000,0.400454,0.916317
3,2021-01-26,676,0,refrig dsrts,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,representative,1,26,0,26,0,0.0,1.000000,0.416125,0.909308
4,2021-01-27,676,3,refrig dsrts,pos subd dairy other,pos dept dairy,other dairy,0,0.0,0,...,representative,2,27,0,27,0,0.0,1.000000,0.431673,0.902030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76044,2023-02-18,962454,3,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,representative,5,18,1,49,1,0.5,0.866025,0.733885,0.679273
76045,2023-02-19,962454,5,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,representative,6,19,1,50,1,0.5,0.866025,0.745438,0.666575
76046,2023-02-20,962454,1,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,representative,0,20,1,51,0,0.5,0.866025,0.756771,0.653680
76047,2023-02-21,962454,6,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,representative,1,21,1,52,0,0.5,0.866025,0.767880,0.640593


In [ ]:
WINDOW_SIZE = 30
STEP_SIZE = 7
SIMILARITY_METHOD_SPEARMAN = "spearman"
SIMILARITY_METHOD_PEARSON = "pearson"
SIMILARITY_METHOD_KENDALL = "kendall"
SIMILARITY_THRESHOLD_SPEARMAN = 0.8
SIMILARITY_THRESHOLD_PEARSON = 0.9
SIMILARITY_THRESHOLD_KENDALL = 0.7

In [14]:
'''
graphs, sim_dfs, df_pivots, window_info = build_dynamic_similarity_graphs(
    df, 
    date_col=DATE_COL, 
    item_col=ITEM_COL, 
    target_col=TARGET_COL, 
    window_size=WINDOW_SIZE, 
    step_size=STEP_SIZE, 
    similarity_method=SIMILARITY_METHOD_SPEARMAN,
    similarity_threshold=SIMILARITY_THRESHOLD_SPEARMAN
)
'''

'\ngraphs, sim_dfs, df_pivots, window_info = build_dynamic_similarity_graphs(\n    df, \n    date_col=DATE_COL, \n    item_col=ITEM_COL, \n    target_col=TARGET_COL, \n    window_size=WINDOW_SIZE, \n    step_size=STEP_SIZE, \n    similarity_method=SIMILARITY_METHOD_SPEARMAN,\n    similarity_threshold=SIMILARITY_THRESHOLD_SPEARMAN\n)\n'

In [15]:
#save_dynamic_graph_plots(graphs, strategy_name=f"Dynamic Spearman Similarity_ws{WINDOW_SIZE}_ss{STEP_SIZE}", base_output_folder="dynamic_similarity_graphs")

In [16]:
graphs, sim_dfs, df_pivots, window_info = build_dynamic_similarity_graphs(
    df, 
    date_col=DATE_COL, 
    item_col=ITEM_COL, 
    target_col=TARGET_COL, 
    window_size=WINDOW_SIZE, 
    step_size=STEP_SIZE, 
    similarity_method=SIMILARITY_METHOD_KENDALL,
    similarity_threshold=SIMILARITY_THRESHOLD_KENDALL
)


Building graph for window: 2021-01-23 00:00:00 to 2021-02-06 00:00:00
Added edge between 676 and 20230 with kendall similarity: 0.7128
Added edge between 692 and 48378 with kendall similarity: 0.7214
Added edge between 47199 and 962453 with kendall similarity: 0.7468
Added edge between 48369 and 103776 with kendall similarity: 0.7279
Added edge between 48369 and 103781 with kendall similarity: 0.7503
Added edge between 51317 and 51320 with kendall similarity: 0.7117
Added edge between 51317 and 608215 with kendall similarity: 0.7312
Added edge between 121853 and 962453 with kendall similarity: 0.7355
Added edge between 270072 and 582895 with kendall similarity: 0.7357
Added edge between 288729 and 606321 with kendall similarity: 0.7536
Number of nodes in the kendall graph: 100
Number of edges in the kendall graph: 10

Building graph for window: 2021-02-22 00:00:00 to 2021-03-08 00:00:00
Number of nodes in the kendall graph: 100
Number of edges in the kendall graph: 0

Building graph f

In [17]:
save_dynamic_graph_plots(graphs, strategy_name=f"Dynamic Kendall Similarity_ws{WINDOW_SIZE}_ss{STEP_SIZE}", base_output_folder="dynamic_similarity_graphs")

Saving 25 dynamic graph plots into dynamic_similarity_graphs\dynamic_kendall_similarity_ws15_ss30...
Graph automatically saved to: dynamic_similarity_graphs\dynamic_kendall_similarity_ws15_ss30\dynamic_kendall_similarity_ws15_ss30_2021-01-23_00-00-00_to_2021-02-06_00-00-00_17_nodes.png
Graph automatically saved to: dynamic_similarity_graphs\dynamic_kendall_similarity_ws15_ss30\dynamic_kendall_similarity_ws15_ss30_2021-02-22_00-00-00_to_2021-03-08_00-00-00_0_nodes.png
Graph automatically saved to: dynamic_similarity_graphs\dynamic_kendall_similarity_ws15_ss30\dynamic_kendall_similarity_ws15_ss30_2021-03-24_00-00-00_to_2021-04-07_00-00-00_2_nodes.png
Graph automatically saved to: dynamic_similarity_graphs\dynamic_kendall_similarity_ws15_ss30\dynamic_kendall_similarity_ws15_ss30_2021-04-23_00-00-00_to_2021-05-07_00-00-00_0_nodes.png
Graph automatically saved to: dynamic_similarity_graphs\dynamic_kendall_similarity_ws15_ss30\dynamic_kendall_similarity_ws15_ss30_2021-05-23_00-00-00_to_2021-